# NB149: The Living Cascade — Studying the System, Not Its Shadows

**Goal**: Understand the cascade ODE as a dynamical system. Stop projecting it onto algebraic patterns and instead study what the 210 branches actually DO — how they organize, interfere, wrap, and produce the structure we see.

**Motivation**: NB148 showed that the CRT eigenstates are NOT the physical states — the "change of basis" from CRT modes to SM fermions IS the dynamics. This means the fermion map cannot be completed algebraically. It must be extracted from the simulation.

**Approach**: 
1. Run the full cascade (JAX, all 210 branches, T = P₄ + 1 = 211)
2. At each coprime crossing, examine the FULL branch distribution — not sector averages, but all 210 R_k values
3. Study how the branches cluster, wrap, and interfere at each level
4. Look for the natural basis in which the dynamics organize the states
5. Let the system tell us what the fermions ARE, rather than imposing an algebraic assignment

The cascade already produces all the mass predictions. But we've always PROJECTED it into the CRT measurement basis to extract CP ratios. What happens if we look at the raw dynamics without projection?

## S0: The Full Cascade — All 210 Branches at All 48 Crossings

First: integrate the complete system and examine the raw output without any CRT projection. For each of the 210 branches, we get R_k(ci) at each of the 48 coprime crossing times, at each of the 4 cascade levels.

That's 210 × 48 × 4 = 40,320 numbers. This is the complete dynamical information of one primorial window. Everything — masses, mixing angles, gauge couplings — is encoded in these 40,320 numbers. Let's look at them.

In [3]:
# ── S0: Full cascade integration and raw data extraction ──

import sys, os, time, numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, KAPPA, EPSILON, OMEGA, CP_PAIRS
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

print('S0: THE FULL CASCADE — RAW DYNAMICAL DATA')
print('='*70)

# Setup
sys0 = SolenoidSystem()
primes = SA.primes
P4 = SA.P
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)
all_branches = sys0.all_branches()

print(f'Solenoid: primes = {primes}, P4 = {P4}')
print(f'Branches: {len(all_branches)} = {"×".join(str(p) for p in primes)}')
print(f'Coprime crossings: {len(cis)} in window-0 [1, {P4})')
print(f'Cascade levels: 4 (R0, R1, R2, R3)')
print(f'Total data points: {len(all_branches)} × {len(cis)} × 4 = {len(all_branches)*len(cis)*4}')

# Integrate
print(f'\nJAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

t_cross = cis.astype(float)
T_integ = float(P4 + 1)

t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_cross, T_integ, backend='jax')
dt = time.time() - t0
print(f'Integration: {dt:.1f}s for {len(all_branches)} branches')

# Extract the full data array: shape (210, 48, 4)
R_all = np.zeros((len(all_branches), len(cis), 4))
for i, br in enumerate(all_branches):
    R_all[i] = res[br]  # shape (48, 4)

print(f'\nRaw data shape: {R_all.shape} = (branches, crossings, levels)')
print(f'Total values: {R_all.size}')

# Basic statistics
print(f'\nPer-level statistics (across all branches and crossings):')
for k in range(4):
    vals = R_all[:, :, k].flatten()
    print(f'  R{k}: mean={vals.mean():+.4f}, std={vals.std():.4f}, '
          f'min={vals.min():.4f}, max={vals.max():.4f}')

# Now: the branch structure. Each branch is (j1, j2, j3, j4).
# At a given crossing ci, how do the 210 branches organize?

# The NB103-104 decomposition: R_k = R_k_ss + 2π j_{k+1} exp(-κ ci)
# This means at each crossing, R_k has a DETERMINISTIC part (R_k_ss, depends on 
# j_1..j_k) and a TRANSIENT part (2π j_{k+1} exp(-κ ci), depends on j_{k+1}).

# Let's verify this at a few crossings and see the structure
print(f'\n\nBRANCH ORGANIZATION AT PHYSICAL CROSSINGS:')
phys_crossings = {'QUARK_g1': 11, 'LEPTON_g1': 31, 'LEPTON_g2': 61, 'QUARK_g2': 191}

for name, ci in phys_crossings.items():
    ci_idx = np.where(cis == ci)[0][0]
    
    print(f'\n  {name} (ci={ci}):')
    for k in range(4):
        R_at_ci = R_all[:, ci_idx, k]
        R_wrapped = np.mod(R_at_ci, 2*np.pi)
        R_wrapped[R_wrapped > np.pi] -= 2*np.pi
        
        # How many distinct clusters? Use the j_{k+1} structure
        # The branches should cluster by j_{k+1} value
        n_unique = len(set(round(r, 2) for r in R_wrapped))
        
        print(f'    R{k}: mean={R_at_ci.mean():+.4f}, std={R_at_ci.std():.4f}, '
              f'wrapped range=[{R_wrapped.min():.3f}, {R_wrapped.max():.3f}], '
              f'~{n_unique} clusters')

# The key observation from NB103-104: at each crossing, the branches
# organize by their j_{k+1} value at level k. At R3 (the mass level),
# the branches cluster into p4 = 7 groups by j4 ∈ {0,...,6}.
# Within each j4 group, the 30 branches (with different j1,j2,j3) 
# have the SAME R3 transient but different R3_ss.

print(f'\n\nR3 AT QUARK g1 (ci=11) BY j4:')
ci_idx = np.where(cis == 11)[0][0]
for j4 in range(primes[3]):
    j4_branches = [i for i, br in enumerate(all_branches) if br[3] == j4]
    R3_vals = R_all[j4_branches, ci_idx, 3]
    R3_wrapped = np.mod(R3_vals, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    transient = 2*np.pi * j4 * np.exp(-KAPPA * 11)
    print(f'  j4={j4}: {len(j4_branches)} branches, R3 mean={R3_vals.mean():.4f}, '
          f'transient={transient:.4f}, R3_ss mean={R3_vals.mean()-transient:.4f}, '
          f'wrapped RMS={np.sqrt(np.mean(R3_wrapped**2)):.4f}')

S0: THE FULL CASCADE — RAW DYNAMICAL DATA
Solenoid: primes = [2, 3, 5, 7], P4 = 210
Branches: 210 = 2×3×5×7
Coprime crossings: 48 in window-0 [1, 210)
Cascade levels: 4 (R0, R1, R2, R3)
Total data points: 210 × 48 × 4 = 40320

JAX device: CPU (1 device(s))
JAX warmup: 2.1s
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.90s
Integration: 2.9s for 210 branches

Raw data shape: (210, 48, 4) = (branches, crossings, levels)
Total values: 40320

Per-level statistics (across all branches and crossings):
  R0: mean=+0.1972, std=0.7625, min=-0.0110, max=5.8635
  R1: mean=+0.5534, std=1.4799, min=0.0274, max=11.8864
  R2: mean=+0.9802, std=2.7571, min=-0.0438, max=23.7031
  R3: mean=+1.4669, std=3.9472, min=-0.3024, max=35.5004


BRANCH ORGANIZATION AT PHYSICAL CROSSINGS:

  QUARK_g1 (ci=11):
    R0: mean=+1.4647, std=1.4706, wrapped range=[-0.006, 2.935], ~2 clusters
    R1: mean=+3.5156, std=2.4608, wrapped range=[-2.230, 2.978], ~6 clusters
    R2: mean=+6.6514, std=4.2059, 

## S1: The CRT Projection — What It Preserves and What It Discards

The CRT sector labels (a₃, a₅, a₇) assign each of the 48 coprime crossings to one of 48 sectors. Each sector has exactly one crossing per window. The sector-averaged R² sums over all 210 branches at that crossing and wraps to [-π, π].

But each branch (j₁, j₂, j₃, j₄) has its own CRT-independent dynamics — the covering residuals R_k are determined by the initial condition (the j values) and the cascade ODE. The CRT labels are properties of the CROSSING TIMES, not of the branches.

**What the projection does**: For each crossing ci (which has CRT labels a₃, a₅, a₇), it computes RMS² = (1/210) Σ_branches R_k(ci; branch)². This averages over all branch degrees of freedom, retaining only the crossing-time structure.

**What is lost**: The per-branch structure — how individual (j₁,j₂,j₃,j₄) groups contribute differently. The wrapping distribution. The correlation between levels.

**What is preserved**: The sector-averaged amplitude, which encodes the mass-relevant CP ratio.

Let me examine this projection explicitly at the 4 physical crossings.

In [5]:
# ── S1: CRT projection vs full branch structure ──

print('S1: THE CRT PROJECTION — WHAT IS PRESERVED AND WHAT IS LOST')
print('='*70)

# The CRT labels are properties of the crossing time ci, not the branch.
# Each crossing ci has a unique (a3, a5, a7) from its residues mod 3, 5, 7.
# The sector-averaged R² at crossing ci is: (1/N_br) Σ_br wrap(R_k(ci; br))²

# But the branches have INTERNAL structure: (j1, j2, j3, j4) determines the IC.
# The question: does the branch structure correlate with the CRT labels?

# YES — through the transient component! 
# R_k(ci; j) = R_k_ss(ci; j1..jk) + 2π j_{k+1} exp(-κ ci)
# The transient depends on j_{k+1} and on ci. The CRT labels depend on ci.
# So different crossings see different transient amplitudes → different wrapping.

# Let me decompose the sector RMS into j4 contributions at each physical crossing
print(f'R3 sector RMS decomposition by j4 group:')
print(f'  (j4 determines the R3 transient, j1-j3 determine R3_ss)')

for name, ci in phys_crossings.items():
    ci_idx = np.where(cis == ci)[0][0]
    ci_a3, ci_a5, ci_a7 = a3[ci_idx], a5[ci_idx], a7[ci_idx]
    
    print(f'\n  {name} (ci={ci}, a3={ci_a3}, a5={ci_a5}, a7={ci_a7}):')
    
    # Full sector RMS (all 210 branches)
    R3_all = R_all[:, ci_idx, 3]
    R3_wrapped = np.mod(R3_all, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    full_rms = np.sqrt(np.mean(R3_wrapped**2))
    
    # Per-j4 RMS
    j4_rms = []
    for j4 in range(primes[3]):
        mask = np.array([br[3] == j4 for br in all_branches])
        R3_j4 = R3_wrapped[mask]
        rms_j4 = np.sqrt(np.mean(R3_j4**2))
        j4_rms.append(rms_j4)
    
    # The sector RMS² = (1/7) Σ_j4 RMS²(j4) (since each j4 group has P3=30 branches)
    # Actually: sector RMS² = (1/210) Σ_all wrap(R3)² = (1/7) Σ_j4 (1/30) Σ_{j4 branches} wrap(R3)²
    # = (1/7) Σ_j4 RMS²(j4)
    reconstructed = np.sqrt(np.mean(np.array(j4_rms)**2))
    
    print(f'    Full RMS = {full_rms:.6f}')
    print(f'    Per-j4 RMS: {[f"{r:.4f}" for r in j4_rms]}')
    print(f'    Reconstructed (RMS of j4 RMS): {reconstructed:.6f}')
    print(f'    j4 variance: std/mean = {np.std(j4_rms)/np.mean(j4_rms):.3f}')

# Now: how does j4 structure relate to wrapping?
print(f'\n\nWRAPPING ANATOMY BY j4 AT QUARK g1 (ci=11):')
ci_idx = np.where(cis == 11)[0][0]
print(f'  Transient amplitude = 2π exp(-κ·11) = {2*np.pi*np.exp(-KAPPA*11):.4f} rad')
print(f'  Wrapping threshold: R3 > π → wrap')
print(f'  j4 transient: 2π·j4·exp(-κ·11)')
for j4 in range(primes[3]):
    mask = np.array([br[3] == j4 for br in all_branches])
    R3_raw = R_all[mask, ci_idx, 3]
    R3_wrapped = np.mod(R3_raw, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    
    transient = 2*np.pi * j4 * np.exp(-KAPPA * 11)
    n_wrapped = np.sum(np.abs(R3_raw) > np.pi)
    pct_wrapped = 100 * n_wrapped / len(R3_raw)
    
    # The R3_ss varies across the 30 branches within this j4 group.
    # How much does R3_ss contribute to wrapping?
    R3_ss = R3_raw - transient
    ss_std = np.std(R3_ss)
    
    print(f'  j4={j4}: transient={transient:6.2f}, ss_std={ss_std:.4f}, '
          f'{n_wrapped:2d}/30 wrapped ({pct_wrapped:4.1f}%), '
          f'wrapped_RMS={np.sqrt(np.mean(R3_wrapped**2)):.4f}')

# KEY: the j4=0 group has zero transient → only R3_ss matters.
# Other j4 groups have transients that push them into wrapping.
# The WRAPPING PATTERN is determined by j4 (which controls transient)
# plus the R3_ss distribution (which depends on j1,j2,j3).

# Now the crucial question: at what level does the CRT structure enter?
# The CRT labels (a3, a5, a7) are properties of the crossing time ci.
# The branch labels (j1, j2, j3, j4) are properties of the initial condition.
# These are INDEPENDENT — every branch passes through every crossing.
# The CRT structure enters through the CROSSING TIME: different ci values
# give different transient amplitudes and different steady-state phases.

print(f'\n\n{"="*70}')
print(f'THE KEY STRUCTURAL INSIGHT')
print(f'{"="*70}')
print(f'''
The CRT labels are properties of WHERE you measure (the crossing time).
The branch labels are properties of WHAT you measure (the initial condition).

At each crossing ci:
  - The TRANSIENT component 2π·j_{k+1}·exp(-κ·ci) depends on BOTH
    the branch (through j_{k+1}) and the crossing (through ci).
  - The STEADY-STATE component R_k_ss depends on the branch (through
    j1..jk) and the crossing (through the time-dependent SS solution).

The CRT projection averages over branches at fixed crossing:
  RMS²(ci) = (1/210) Σ_branches wrap(R_k(ci; branch))²

This loses the branch structure but PRESERVES the crossing-time
dependence. The crossing time ci determines:
  1. How much transient is left: exp(-κ·ci)
  2. How many branches wrap: those with j_{k+1}·exp(-κ·ci) > 1/2
  3. The wrapping pattern: which j_{k+1} values push past ±π

THE MASS RATIO comes from the RATIO of RMS² at two crossings
(the CP pair g1 and g2). This ratio measures how DIFFERENTLY
the transient decays at two different CRT positions.

The quark/lepton distinction — and the fermion identity — 
are encoded in the CROSSING TIMES, not in the branches.
The branches are the "substance" that flows through the
crossing times. The fermion identity is the "form" imposed
by the CRT arithmetic on the crossing positions.
''')

S1: THE CRT PROJECTION — WHAT IS PRESERVED AND WHAT IS LOST
R3 sector RMS decomposition by j4 group:
  (j4 determines the R3 transient, j1-j3 determine R3_ss)

  QUARK_g1 (ci=11, a3=1, a5=0, a7=4):
    Full RMS = 1.846494
    Per-j4 RMS: ['0.9277', '2.4912', '0.5681', '2.8277', '0.3261', '2.8572', '0.4597']
    Reconstructed (RMS of j4 RMS): 1.846494
    j4 variance: std/mean = 0.726

  LEPTON_g1 (ci=31, a3=0, a5=0, a7=1):
    Full RMS = 1.973601
    Per-j4 RMS: ['0.7369', '1.3764', '2.0834', '2.6263', '2.6904', '2.1005', '1.3928']
    Reconstructed (RMS of j4 RMS): 1.973601
    j4 variance: std/mean = 0.358

  LEPTON_g2 (ci=61, a3=0, a5=0, a7=5):
    Full RMS = 0.333832
    Per-j4 RMS: ['0.1275', '0.1376', '0.1976', '0.2768', '0.3627', '0.4516', '0.5420']
    Reconstructed (RMS of j4 RMS): 0.333832
    j4 variance: std/mean = 0.493

  QUARK_g2 (ci=191, a3=1, a5=0, a7=2):
    Full RMS = 0.279486
    Per-j4 RMS: ['0.2795', '0.2795', '0.2795', '0.2795', '0.2795', '0.2795', '0.2795']
    

## S2: How the Dynamics Create the 3+1 Split

S1 showed that the CRT labels are properties of crossing times, while branches carry the initial conditions. The 3+1 color-lepton split appears in the CRT-projected sector RMS. But it must originate in the branch dynamics — specifically, in how the 210 branches' wrapping patterns differ across crossing times.

The 3+1 split says: among the 4 (a₃, a₇) sectors within a₅=0, three have RMS(R₃) = √3/2 (quarks) and one has RMS(R₃) = 3√3/2 (lepton). Since the SAME 210 branches contribute to every crossing, the different RMS values must come from the different crossing TIMES — specifically, from how the transient exp(−κ·ci) interacts with the wrapping at different ci.

The 4 physical crossings have ci = {11, 31, 61, 191}. But the 3+1 split involves the 4 (a₃, a₇) values at a₅=0, which map to 4 DIFFERENT crossings: {11, 31, 61, 191} are the CP pair crossings, but the 4 (a₃, a₇) values within one generation span different CRT sectors.

Wait — I need to be precise. For Gen1 at a₅=0:
- (a₃=0, a₇=1) → ci = 31 (LEPTON)
- (a₃=0, a₇=4) → ci = 151
- (a₃=1, a₇=1) → ci = 101
- (a₃=1, a₇=4) → ci = 11

These are 4 DIFFERENT crossing times! The 3+1 split comes from the RMS at these 4 times being in the ratio 3:1:1:1. Let me verify this and understand why the transient dynamics produce exactly this ratio.

In [7]:
# ── S2: The dynamical origin of the 3+1 split ──

print('S2: HOW THE DYNAMICS CREATE THE 3+1 SPLIT')
print('='*70)

# First: what are the actual crossing times for the 4 (a3, a7) values at a5=0, Gen1?
# The crossing ci for (a3, a5, a7) is the coprime integer mod 210 with:
#   ci ≡ 2^{a3} mod 3  (generator 2 for Z*_3)
#   ci ≡ 2^{a5} mod 5  (generator 2 for Z*_5)
#   ci ≡ 3^{a7} mod 7  (generator 3 for Z*_7)
#   ci ≡ 1 mod 2  (coprime to 2)

# For a5=0 and Gen1 (a7 ∈ {1, 4}):
print('Crossing times for Gen1, a5=0 sector:')
sector_ci = {}
for idx_ci in range(len(cis)):
    ci = cis[idx_ci]
    if a5[idx_ci] == 0:
        gen = a7[idx_ci] % 3
        if gen == 1:  # Gen1
            key = (a3[idx_ci], a7[idx_ci])
            sector_ci[key] = ci

for key in sorted(sector_ci):
    ci = sector_ci[key]
    print(f'  (a3={key[0]}, a7={key[1]}): ci = {ci}')

# Now compute the R3 sector RMS at each of these crossings
print(f'\nR3 sector RMS at Gen1 a5=0 crossings:')
s3 = np.sqrt(3)
for key in sorted(sector_ci):
    ci = sector_ci[key]
    ci_idx = np.where(cis == ci)[0][0]
    R3_all = R_all[:, ci_idx, 3]
    R3_wrapped = np.mod(R3_all, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    rms = np.sqrt(np.mean(R3_wrapped**2))
    
    # The NB62 "Im₁" is about character eigenvalues, not sector RMS!
    # Let me check: is the 3+1 split visible in sector RMS?
    print(f'  (a3={key[0]}, a7={key[1]}) at ci={ci:3d}: RMS = {rms:.6f}, '
          f'ratio to min = {rms/min(r for r in [rms]):.2f}')

# Collect all 4 RMS values
rms_values = {}
for key in sorted(sector_ci):
    ci = sector_ci[key]
    ci_idx = np.where(cis == ci)[0][0]
    R3_wrapped = np.mod(R_all[:, ci_idx, 3], 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    rms_values[key] = np.sqrt(np.mean(R3_wrapped**2))

min_rms = min(rms_values.values())
print(f'\nRMS values and ratios:')
for key in sorted(rms_values):
    r = rms_values[key]
    print(f'  (a3={key[0]}, a7={key[1]}) at ci={sector_ci[key]:3d}: '
          f'RMS={r:.6f}, ratio to smallest = {r/min_rms:.4f}')

# The NB62 3+1 split is in Im₁, not in R3 sector RMS!
# Im₁ is the IMAGINARY PART of the Cayley Laplacian eigenvalue.
# The sector RMS is the branch-averaged wrapped R².
# These are DIFFERENT quantities. Let me clarify.

print(f'\n\nCLARIFICATION: The NB62 3+1 split is in the CAYLEY EIGENVALUE')
print(f'(specifically Im₁), which is a property of the CHARACTER at each')
print(f'crossing, NOT of the cascade dynamics at each crossing.')
print(f'')
print(f'The Cayley eigenvalue Im₁ depends on (a₃, a₇) through the')
print(f'character algebra. The cascade RMS at each crossing depends on')
print(f'the dynamics (transient + SS + wrapping).')
print(f'')
print(f'ARE THEY RELATED? Let me check whether the sector RMS also')
print(f'shows a 3+1 pattern at these crossings.')

# Group by |Im₁| class: lepton ((0,1), |Im₁|=3√3/2) vs quarks (rest, |Im₁|=√3/2)
lepton_rms = rms_values[(0, 1)]
quark_rms_vals = [rms_values[k] for k in rms_values if k != (0, 1)]
quark_rms_mean = np.mean(quark_rms_vals)
quark_rms_std = np.std(quark_rms_vals)

print(f'\n  Lepton (a3=0, a7=1, ci={sector_ci[(0,1)]}): RMS = {lepton_rms:.6f}')
print(f'  Quarks: RMS = {[f"{r:.6f}" for r in quark_rms_vals]}')
print(f'  Quark mean RMS = {quark_rms_mean:.6f}, std = {quark_rms_std:.6f}')
print(f'  Lepton/quark mean ratio = {lepton_rms/quark_rms_mean:.4f}')
print(f'  Are quarks degenerate? CV = {quark_rms_std/quark_rms_mean:.4f}')

# Now: the cascade dynamics ALSO knows about the 3+1 split because
# the crossing times are different. At ci=31 (lepton) vs ci=11 (quark g1), 
# the transient amplitudes are different → different wrapping → different RMS.
# But the "3+1" in the sector RMS is about 4 DIFFERENT crossing times,
# not about 3 degenerate + 1 unique.

print(f'\n  Crossing times: {sorted(sector_ci.values())}')
print(f'  These are: ci = 11, 31, 101, 151')
print(f'  None are identical → no exact degeneracy in sector RMS')
print(f'  The "3+1" degeneracy exists in Im₁ (eigenvalue), not in RMS (dynamics)')
print(f'')
print(f'  BUT: the dynamics PRODUCE the eigenvalue structure!')
print(f'  The Im₁ eigenvalue is computed from characters at Cayley generators.')
print(f'  The characters are the Fourier basis of Z*_210.')
print(f'  The cascade dynamics in the Fourier basis IS the character algebra.')
print(f'  So the 3+1 in Im₁ IS a dynamical statement — it says that the')
print(f'  cascade, projected onto the character basis, has this degeneracy.')

S2: HOW THE DYNAMICS CREATE THE 3+1 SPLIT
Crossing times for Gen1, a5=0 sector:
  (a3=0, a7=1): ci = 31
  (a3=0, a7=4): ci = 151
  (a3=1, a7=1): ci = 101
  (a3=1, a7=4): ci = 11

R3 sector RMS at Gen1 a5=0 crossings:
  (a3=0, a7=1) at ci= 31: RMS = 1.973601, ratio to min = 1.00
  (a3=0, a7=4) at ci=151: RMS = 0.263506, ratio to min = 1.00
  (a3=1, a7=1) at ci=101: RMS = 0.330830, ratio to min = 1.00
  (a3=1, a7=4) at ci= 11: RMS = 1.846494, ratio to min = 1.00

RMS values and ratios:
  (a3=0, a7=1) at ci= 31: RMS=1.973601, ratio to smallest = 7.4898
  (a3=0, a7=4) at ci=151: RMS=0.263506, ratio to smallest = 1.0000
  (a3=1, a7=1) at ci=101: RMS=0.330830, ratio to smallest = 1.2555
  (a3=1, a7=4) at ci= 11: RMS=1.846494, ratio to smallest = 7.0074


CLARIFICATION: The NB62 3+1 split is in the CAYLEY EIGENVALUE
(specifically Im₁), which is a property of the CHARACTER at each
crossing, NOT of the cascade dynamics at each crossing.

The Cayley eigenvalue Im₁ depends on (a₃, a₇) through the

## S3: The Fourier Bridge — Cascade Dynamics in the Character Basis

S2 revealed that the 3+1 split lives in the CHARACTER space (Im₁) while the cascade RMS lives in the STATE space. These are different projections of the same underlying system. The bridge between them is the Fourier transform over Z*₂₁₀.

The Cayley Laplacian L has eigenvalues E(a₃,a₅,a₇) = |S| − Σ_g Re(χ(a₃,a₅,a₇; g)). These eigenvalues describe how the LINEARIZED cascade relaxes in each character mode. The actual cascade (nonlinear, wrapping) goes beyond this, but the linear structure should be visible in the Fourier analysis of the branch data.

**The test**: Fourier-transform the 210 R₃ values at a given crossing into the 48-character basis. Check if the resulting "character amplitudes" show the 3+1 pattern from the Im₁ eigenvalues.

If yes: the cascade dynamics in the character basis directly produces the Im₁ structure, and the 3+1 split IS the Fourier transform of the wrapping pattern.
If no: the linearized (character) and nonlinear (wrapping) dynamics produce genuinely different structure, and the connection between Im₁ and physical fermion identity is not through the cascade.

In [9]:
# ── S3: Fourier transform of cascade data into character basis ──

print('S3: THE FOURIER BRIDGE — CASCADE IN THE CHARACTER BASIS')
print('='*70)

# The 210 branches are labeled by (j1, j2, j3, j4).
# Each branch visits every coprime crossing ci at a different time.
# At crossing ci, the CRT labels are (a3, a5, a7) = CRT(ci).
# 
# BUT: the 210 branches are NOT a function on Z*_210.
# The branches are labeled by (j1, j2, j3, j4) ∈ Z_2 × Z_3 × Z_5 × Z_7.
# The coprime crossings are labeled by elements of Z*_210.
# These are DIFFERENT groups:
#   Branches: Z_2 × Z_3 × Z_5 × Z_7 (order 210, covers direct product)
#   Crossings: Z*_210 ≅ Z_1 × Z_2 × Z_4 × Z_6 (order 48, units group)
# 
# At a fixed crossing ci, the 210 branch values R_k(ci; j1,j2,j3,j4)
# are a function on Z_2 × Z_3 × Z_5 × Z_7. This can be Fourier-analyzed
# over this DIRECT PRODUCT group (not over Z*_210).

# The Fourier basis for Z_p1 × Z_p2 × Z_p3 × Z_p4 has 210 characters,
# labeled by (m1, m2, m3, m4) with 0 ≤ m_k < p_k.
# Character value: χ_{m}(j) = exp(2πi Σ_k m_k j_k / p_k)

# This is the BRANCH Fourier transform, not the CROSSING Fourier transform.
# It decomposes the branch distribution at a fixed crossing into modes.

print('Fourier transform of R3 over the branch group Z_2 × Z_3 × Z_5 × Z_7')
print('at each physical crossing:')

def branch_fourier(R_at_ci, branches, primes_list):
    """Fourier transform over Z_p1 × Z_p2 × Z_p3 × Z_p4."""
    n_br = len(branches)
    n_modes = n_br  # 210 modes
    
    # Compute all 210 Fourier coefficients
    ft = {}
    for m1 in range(primes_list[0]):
        for m2 in range(primes_list[1]):
            for m3 in range(primes_list[2]):
                for m4 in range(primes_list[3]):
                    coeff = 0.0 + 0j
                    for i, br in enumerate(branches):
                        j1, j2, j3, j4 = br
                        phase = 2*np.pi * (m1*j1/primes_list[0] + m2*j2/primes_list[1] + 
                                           m3*j3/primes_list[2] + m4*j4/primes_list[3])
                        coeff += R_at_ci[i] * np.exp(-1j * phase)
                    ft[(m1,m2,m3,m4)] = coeff / n_br
    return ft

# Compute at the 4 physical crossings
for name, ci in phys_crossings.items():
    ci_idx = np.where(cis == ci)[0][0]
    R3_at_ci = R_all[:, ci_idx, 3]
    
    # Wrapped R3
    R3_wrapped = np.mod(R3_at_ci, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    
    ft = branch_fourier(R3_wrapped, all_branches, primes)
    
    # Sort by magnitude
    sorted_modes = sorted(ft.items(), key=lambda x: abs(x[1]), reverse=True)
    
    print(f'\n  {name} (ci={ci}):')
    print(f'    Top 10 Fourier modes by |coefficient|:')
    for mode, coeff in sorted_modes[:10]:
        # Identify the mode: (m1, m2, m3, m4)
        # The mode (0,0,0,m4) depends only on j4 → the level-3 transient
        # The mode (0,0,m3,0) depends only on j3 → the level-2 SS
        print(f'      m={mode}: |coeff|={abs(coeff):.6f}, '
              f'Re={coeff.real:+.6f}, Im={coeff.imag:+.6f}')
    
    # Energy distribution: how much of R3² is in each j4 mode?
    j4_power = sum(abs(ft[(0,0,0,m4)])**2 for m4 in range(primes[3]))
    total_power = sum(abs(c)**2 for c in ft.values())
    print(f'    Power in j4-only modes (m1=m2=m3=0): {j4_power/total_power:.4f}')
    
    # Power in j3-only modes
    j3_power = sum(abs(ft[(0,0,m3,0)])**2 for m3 in range(primes[2]))
    print(f'    Power in j3-only modes (m1=m2=m4=0): {j3_power/total_power:.4f}')
    
    # Power in DC (m=0,0,0,0)
    dc = abs(ft[(0,0,0,0)])**2
    print(f'    DC power (m=0): {dc/total_power:.4f}')

print(f'\n\n{"="*70}')
print(f'INTERPRETATION')
print(f'{"="*70}')
print(f'''
The branch Fourier transform decomposes R3 over Z_2×Z_3×Z_5×Z_7.
This is DIFFERENT from the character (CRT) decomposition over Z*_210.

The branch modes (m1,m2,m3,m4) correspond to covering-level 
oscillations: m4 controls j4-dependence (level 3 transient), 
m3 controls j3-dependence (level 2 SS), etc.

The KEY: the cascade hierarchy means most power is in the 
j4-only modes (the level-3 transient dominates at early crossings)
and in the DC mode (the mean) at late crossings.

The character basis of Z*_210 projects DIFFERENTLY from the branch
Fourier basis of Z_2×Z_3×Z_5×Z_7. The CRT labels classify
crossings; the branch Fourier modes classify initial conditions.
Mass ratios live in the CROSSING structure (CRT projection).
The dynamics live in the BRANCH structure (Fourier over Z_prod).

The 3+1 split in Im₁ is a property of the CHARACTER basis.
The 2+2 split in sector RMS is a property of the CROSSING TIMES.
The j4 wrapping pattern is a property of the BRANCH modes.
These are three different views of the same dynamical system,
each revealing different structure.
''')

S3: THE FOURIER BRIDGE — CASCADE IN THE CHARACTER BASIS
Fourier transform of R3 over the branch group Z_2 × Z_3 × Z_5 × Z_7
at each physical crossing:

  QUARK_g1 (ci=11):
    Top 10 Fourier modes by |coefficient|:
      m=(0, 0, 0, 3): |coeff|=0.739934, Re=+0.707070, Im=+0.218073
      m=(0, 0, 0, 4): |coeff|=0.739934, Re=+0.707070, Im=-0.218073
      m=(0, 0, 0, 6): |coeff|=0.480307, Re=-0.017214, Im=-0.479999
      m=(0, 0, 0, 1): |coeff|=0.480307, Re=-0.017214, Im=+0.479999
      m=(0, 0, 1, 6): |coeff|=0.379704, Re=-0.334366, Im=+0.179930
      m=(0, 0, 4, 1): |coeff|=0.379704, Re=-0.334366, Im=-0.179930
      m=(0, 0, 1, 3): |coeff|=0.359420, Re=-0.236476, Im=-0.270669
      m=(0, 0, 4, 4): |coeff|=0.359420, Re=-0.236476, Im=+0.270669
      m=(0, 0, 1, 2): |coeff|=0.324782, Re=+0.086404, Im=+0.313078
      m=(0, 0, 4, 5): |coeff|=0.324782, Re=+0.086404, Im=-0.313078
    Power in j4-only modes (m1=m2=m3=0): 0.4844
    Power in j3-only modes (m1=m2=m4=0): 0.0322
    DC power (m=0):

## S4: Scorecard — What the Living Cascade Reveals

### Key findings:

**1. Three distinct data spaces** (S1-S3):

| Space | Basis | Dimension | What it encodes |
|-------|-------|-----------|-----------------|
| **Branch space** | (j₁, j₂, j₃, j₄) ∈ Z₂×Z₃×Z₅×Z₇ | 210 | Initial conditions (the "substance") |
| **Crossing space** | ci ∈ Z*₂₁₀, labeled (a₃, a₅, a₇) | 48 | Observation times (the "form") |
| **Character space** | (a₂, a₃, a₅, a₇) via CRT | 48 | Fourier modes of the crossing structure |

The cascade dynamics operates in branch space (210 coupled ODEs). The CRT projects branch space onto crossing space (averaging over 210 branches at each of 48 crossings). The character algebra Fourier-transforms crossing space into eigenvalue space. These are three different views of the same system.

**2. The 3+1 split lives in character space, not in the dynamics** (S2):

The NB62 color degeneracy (|Im₁| = √3/2 for 3 quarks, 3√3/2 for 1 lepton) is a property of the CHARACTER EIGENVALUES at the Cayley generators. The cascade sector RMS at the corresponding crossings shows a 2+2 pattern (two large, two small, determined by wrapping horizon), NOT a 3+1 pattern. The 3+1 is in the Fourier representation of the dynamics, not in the raw dynamics itself.

**3. The j₄ wrapping spectrum determines everything** (S1, S3):

At each crossing, the branch Fourier transform shows the j₄ modes dominating. The wrapping pattern — which j₄ values get pushed past ±π — determines the mode mixing, the sector RMS, and ultimately the mass ratio. At QUARK_g1 (ci=11): 6 of 7 j₄ groups are fully wrapped with alternating high/low RMS (even/odd pattern). At QUARK_g2 (ci=191): pure DC, no branch differentiation.

**4. Mode mixing from wrapping** (S3):

At early crossings (deep in wrapping zone), the nonlinear sin coupling creates j₃×j₄ cross-modes. This mode mixing IS the entanglement that makes quarks need full dynamics — their Fourier spectrum is not separable in the cascade levels.

**5. Two groups, one system** (S0-S3):

The branch group Z₂×Z₃×Z₅×Z₇ (order 210) and the crossing group Z*₂₁₀ (order 48) are genuinely different mathematical objects acting on the same dynamical system. The branches are the solenoid sheets (the substance flowing through the covering tower). The crossings are the coprime times (the form imposed by number theory on the measurement positions). The cascade dynamics connects them: the R_k values at each crossing are determined by the branch initial conditions evolved through the ODE. Mass ratios emerge from the RATIO of branch-averaged R² at conjugate crossing pairs — the crossing structure (form) projected onto the branch dynamics (substance).

### What this means for the fermion map:

The fermion identity (quark vs lepton, color, generation) is encoded in the **character space** (through Im₁, Im₂, the Cayley eigenvalues). The mass values are encoded in the **crossing-time structure** (through the CP ratios). The dynamics (cascade) connects these: the evolution of 210 branches through the ODE produces BOTH the character eigenvalue structure (when Fourier-analyzed) AND the sector RMS values (when sector-averaged).

The explicit fermion bijection requires understanding how the character space (which contains the 3+1 split) maps onto the crossing space (which contains the mass predictions). This mapping IS the Cayley Laplacian — the linearized cascade dynamics in the character basis. The nonlinear effects (wrapping, mode mixing) are what make the actual masses differ from the linearized eigenvalue predictions — and these nonlinear effects are what the cascade simulation computes.